## Dubai Real Estate Machine Learning Pipeline

This section introduces a comprehensive, object-oriented machine learning pipeline for analyzing Dubai Land Department (DLD) property transaction data. It includes modules for data loading, preprocessing, model training and tuning (Random Forest for both regression and classification), and visualization of results.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import logging
import os
import sys
from typing import Dict, List, Tuple, Any, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# ==============================================================================
# LOGGING CONFIGURATION
# ==============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("DLD_ML_Pipeline")


# ==============================================================================
# DATA LOADER CLASS
# ==============================================================================
class DataLoader:
    """Handles secure data ingestion and initial format integrity checks."""

    def __init__(self, file_path: str, nrows: Optional[int] = None) -> None:
        """Initializes the DataLoader with a specified file path and optional number of rows to load.

        Args:
            file_path: The local filesystem path to the target CSV file.
            nrows: The number of rows to read from the CSV file. If None, reads all rows.
        """
        self.file_path = file_path
        self.nrows = nrows

    def load_data(self) -> pd.DataFrame:
        """Loads a CSV file into a pandas DataFrame with error handling.

        Returns:
            pd.DataFrame: Loaded dataset.

        Raises:
            FileNotFoundError: If the designated file does not exist.
            ValueError: If the file is empty or corrupted.
        """
        logger.info(f"Attempting to ingest dataset from: {self.file_path}")
        try:
            if not os.path.exists(self.file_path):
                raise FileNotFoundError(f"File not found at {self.file_path}")

            df = pd.read_csv(self.file_path, nrows=self.nrows)
            if df.empty:
                raise ValueError("The ingested dataset is empty.")

            logger.info(
                f"Data ingested successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns."
            )
            return df
        except Exception as e:
            logger.error(f"Critical error during data loading execution: {str(e)}")
            raise


# ==============================================================================
# PROPERTY PIPELINE TRANSFORMATION CLASS
# ==============================================================================
class PropertyPipeline:
    """Manages preprocessing state, features isolation, and train-test splits."""

    def __init__(
        self,
        target_reg: str = "actual_worth",
        target_clf: str = "property_usage_en",
        random_state: int = 42,
    ) -> None:
        """Initializes structural properties and explicit randomness guidelines.

        Args:
            target_reg: Field name for the continuous regression target.
            target_clf: Field name for the categorical classification target.
            random_state: Fixed random seed for pipeline reproducibility.
        """
        self.target_reg = target_reg
        self.target_clf = target_clf
        self.random_state = random_state

        self.num_cols: List[str] = []
        self.cat_cols: List[str] = []
        self.feature_cols: List[str] = []
        self.transformed_feature_names: List[str] = []

        self.preprocessor: Optional[ColumnTransformer] = None
        self.label_encoder: LabelEncoder = LabelEncoder()

    def build_preprocessor(self, df: pd.DataFrame) -> None:
        """Identifies columns dynamically and builds the ColumnTransformer pipeline.

        Args:
            df: Raw pandas DataFrame representing complete ingestion space.
        """
        logger.info("Initializing ColumnTransformer configuration rules...")

        # Isolate baseline feature inputs excluding target parameters
        self.feature_cols = [
            col
            for col in df.columns
            if col not in [self.target_reg, self.target_clf]
        ]

        # Isolate numerical and categorical spaces automatically
        self.num_cols = (
            df[self.feature_cols]
            .select_dtypes(include=[np.number])
            .columns.tolist()
        )
        self.cat_cols = (
            df[self.feature_cols]
            .select_dtypes(exclude=[np.number])
            .columns.tolist()
        )

        logger.info(f"Detected numerical features: {self.num_cols}")
        logger.info(f"Detected categorical features: {self.cat_cols}")

        # Enforce exact design constraints specified by Member 2
        self.preprocessor = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), self.num_cols),
                (
                    "cat",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                    self.cat_cols,
                ),
            ]
        )

    def extract_feature_names(self, X_train: pd.DataFrame) -> None:
        """Extracts and maps engineered post-OneHotEncoded column headers.

        Args:
            X_train: Predictor training matrix.
        """
        if self.preprocessor is None:
            raise ValueError("Preprocessor missing framework instantiation.")

        self.preprocessor.fit(X_train)
        cat_encoder = self.preprocessor.named_transformers_["cat"]
        encoded_cat_cols = cat_encoder.get_feature_names_out(
            self.cat_cols
        ).tolist()
        self.transformed_feature_names = self.num_cols + encoded_cat_cols
        logger.info(
            f"Feature spaces mapped. Total post-encoded dimensions: {len(self.transformed_feature_names)}"
        )

    def prepare_regression_split(
        self, df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
        """Generates an 80/20 train-test split optimized for regression modeling."""
        X = df[self.feature_cols]
        y = df[self.target_reg]
        return train_test_split(
            X, y, test_size=0.20, random_state=self.random_state
        )

    def prepare_classification_split(
        self, df: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, List[str]]:
        """Encodes target strings and handles 80/20 split for classification.

        Args:
            df: Input target DataFrame.

        Returns:
            Tuple: X_train, X_test, y_train, y_test arrays alongside class labels string arrays.
        """
        X = df[self.feature_cols]
        y_raw = df[self.target_clf]

        # Standardizing nominal classes cleanly to deterministic integer keys
        y_encoded = self.label_encoder.fit_transform(y_raw)
        class_names = self.label_encoder.classes_.astype(str).tolist()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y_encoded, test_size=0.20, random_state=self.random_state
        )
        return X_train, X_test, y_train, y_test, class_names


# ==============================================================================
# MODEL TRAINING & TUNING MODULE CLASS
# ==============================================================================
class ModelTrainer:
    """Manages baseline execution and RandomizedSearchCV hyperparameter loops."""

    def __init__(
        self,
        preprocessor: ColumnTransformer, random_state: int = 42
    ) -> None:
        """Initializes the trainer module with general tuning parameters.

        Args:
            preprocessor: ColumnTransformer tracking preprocessing configurations.
            random_state: Uniform random tracking variable.
        """
        self.preprocessor = preprocessor
        self.random_state = random_state

        # Hyperparameter grid matching architectural guidelines exactly
        self.param_dist = {
            "model__n_estimators": [50, 100],
            "model__max_depth": [10, 20, None],
            "model__min_samples_split": [2, 5],
            "model__max_features": ["sqrt", "log2"],
        }

    def train_regressor(
        self,
        X_train: pd.DataFrame,
        X_test: pd.DataFrame,
        y_train: pd.Series,
        y_test: pd.Series,
    ) -> Tuple[Pipeline, Dict[str, Any]]:
        """Executes the complete baseline and hyperparameter optimization for regression.

        Args:
            X_train: Regression training features DataFrame.
            X_test: Regression testing features DataFrame.
            y_train: Regression training targets Series.
            y_test: Regression testing targets Series.

        Returns:
            Tuple: Configured best Pipeline and a structured evaluation dictionary.
        """
        logger.info("Initializing Regressor evaluation routines...")

        # 1. Baseline Model Execution
        base_pipe = Pipeline(
            steps=[
                ("preprocessor", self.preprocessor),
                (
                    "model",
                    RandomForestRegressor(
                        n_estimators=100,
                        random_state=self.random_state,
                        n_jobs=1, # Reduced n_jobs to 1 for stability
                    ),
                ),
            ]
        )
        base_pipe.fit(X_train, y_train)
        y_pred_base = base_pipe.predict(X_test)

        # 2. Hyperparameter Tuning via RandomizedSearchCV
        tune_pipe = Pipeline(
            steps=[
                ("preprocessor", self.preprocessor),
                (
                    "model",
                    RandomForestRegressor(
                        random_state=self.random_state, n_jobs=1 # Reduced n_jobs to 1
                    ),
                ),
            ]
        )
        search = RandomizedSearchCV(
            estimator=tune_pipe,
            param_distributions=self.param_dist,
            n_iter=5, # Reduced for faster execution
            cv=3,
            scoring="neg_root_mean_squared_error",
            random_state=self.random_state,
            n_jobs=1, # Reduced n_jobs to 1
        )
        logger.info("Running Regressor RandomizedSearchCV (5 Iterations)...")
        search.fit(X_train, y_train)
        best_pipe = search.best_estimator_
        y_pred_tuned = best_pipe.predict(X_test)

        # Structured validation reporting dictionary
        metrics = {
            "Baseline": {
                "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_base)),
                "MAE": mean_absolute_error(y_test, y_pred_base),
                "R2": r2_score(y_test, y_pred_base),
            },
            "Tuned": {
                "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_tuned)),
                "MAE": mean_absolute_error(y_test, y_pred_tuned),
                "R2": r2_score(y_test, y_pred_tuned),
            },
            "Best_Params": search.best_params_,
        }
        return best_pipe, metrics

    def train_classifier(
        self,
        X_train: pd.DataFrame,
        X_test: pd.DataFrame,
        y_train: np.ndarray,
        y_test: np.ndarray,
    ) -> Tuple[Pipeline, np.ndarray, np.ndarray, Dict[str, Any]]:
        """Executes baseline setup, search iterations, and predicts outcome probability dimensions.

        Args:
            X_train: Classification training features DataFrame.
            X_test: Classification testing features DataFrame.
            y_train: Classification training target arrays.
            y_test: Classification testing target arrays.

        Returns:
            Tuple: Optimized Pipeline, class predictions, class probabilities matrix, and metrics.
        """
        logger.info("Initializing Classifier evaluation routines...")

        # 1. Baseline Model Execution
        base_pipe = Pipeline(
            steps=[
                ("preprocessor", self.preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=100,
                        random_state=self.random_state,
                        n_jobs=1, # Reduced n_jobs to 1 for stability
                    ),
                ),
            ]
        )
        base_pipe.fit(X_train, y_train)
        y_pred_base = base_pipe.predict(X_test)

        # 2. Hyperparameter Tuning via RandomizedSearchCV
        tune_pipe = Pipeline(
            steps=[
                ("preprocessor", self.preprocessor),
                (
                    "model",
                    RandomForestClassifier(
                        random_state=self.random_state, n_jobs=1 # Reduced n_jobs to 1
                    ),
                ),
            ]
        )
        search = RandomizedSearchCV(
            estimator=tune_pipe,
            param_distributions=self.param_dist,
            n_iter=5, # Reduced for faster execution
            cv=3,
            scoring="f1_weighted",
            random_state=self.random_state,
            n_jobs=1, # Reduced n_jobs to 1
        )
        logger.info("Running Classifier RandomizedSearchCV (5 Iterations)...")
        search.fit(X_train, y_train)
        best_pipe = search.best_estimator_

        y_pred_tuned = best_pipe.predict(X_test)
        y_prob_tuned = best_pipe.predict_proba(X_test)

        metrics = {
            "Baseline": {
                "Accuracy": accuracy_score(y_test, y_pred_base),
                "F1_Weighted": f1_score(
                    y_test, y_pred_base, average="weighted"
                ),
            },
            "Tuned": {
                "Accuracy": accuracy_score(y_test, y_pred_tuned),
                "F1_Weighted": f1_score(
                    y_test, y_pred_tuned, average="weighted"
                ),
            },
            "Best_Params": search.best_params_,
        }
        return best_pipe, y_pred_tuned, y_prob_tuned, metrics


# ==============================================================================
# DECOUPLED VISUALIZATION DATA GENERATION CLASS
# ==============================================================================
class Visualizer:
    """Decoupled graphing engine for production-ready model evaluation plots."""

    @staticmethod
    def plot_feature_importance(
        model_pipeline: Pipeline, feature_names: List[str], filename: str
    ) -> None:
        """Saves a horizontal bar chart displaying the top 15 most important features.

        Args:
            model_pipeline: Trained Scikit-Learn Pipeline object containing the regressor.
            feature_names: List of strings tracking parsed post-encoded feature names.
            filename: Active file target path destination.
        """
        logger.info("Generating Top 15 Feature Importances horizontal visualization...")
        try:
            importances = model_pipeline.named_steps["model"].feature_importances_
            indices = np.argsort(importances)[::-1][:15]

            plt.figure(figsize=(11, 7))
            sns.barplot(
                x=importances[indices],
                y=np.array(feature_names)[indices],
                palette="viridis",
                hue=np.array(feature_names)[indices],
                legend=False,
            )
            plt.title(
                "Top 15 Feature Importances Driving Property Pricing (DLD Analysis)",
                fontsize=13,
                fontweight="bold",
                pad=15,
            )
            plt.xlabel("Relative Importance Metric Score", fontsize=11)
            plt.ylabel("Property Structural Characteristics", fontsize=11)
            plt.grid(axis="x", linestyle="--", alpha=0.5)
            plt.tight_layout()
            plt.savefig(filename, dpi=300)
            plt.close()
            logger.info(f"Feature importance visualization saved successfully to {filename}")
        except Exception as e:
            logger.error(f"Failed to generate feature importance plot: {str(e)}")

    @staticmethod
    def plot_confusion_matrix(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        class_names: List[str],
        filename: str,
    ) -> None:
        """Generates and saves a publication-quality confusion matrix heatmap.

        Args:
            y_true: Ground truth target vector array.
            y_pred: Predicted label outcome target array.
            class_names: Text string classifications for data mapping labels.
            filename: Targeted saving destination file string path.
        """
        logger.info("Generating Confusion Matrix Heatmap...")
        try:
            cm = confusion_matrix(y_true, y_pred)
            plt.figure(figsize=(8, 6))
            sns.heatmap(
                cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                xticklabels=class_names,
                yticklabels=class_names,
                cbar=True,
                square=True,
            )
            plt.title(
                "Confusion Matrix: Property Usage Classification",
                fontsize=13,
                fontweight="bold",
                pad=15,
            )
            plt.xlabel("Predicted Categorical Label", fontsize=11)
            plt.ylabel("True Empirical Target Label", fontsize=11)
            plt.tight_layout()
            plt.savefig(filename, dpi=300)
            plt.close()
            logger.info(f"Confusion Matrix graph saved successfully to {filename}")
        except Exception as e:
            logger.error(f"Failed to generate confusion matrix visualization: {str(e)}")

    @staticmethod
    def plot_roc_curves(
        y_true: np.ndarray,
        y_prob: np.ndarray,
        class_names: List[str],
        filename: str,
    ) -> None:
        """Plots multi-class (One-vs-Rest) or binary ROC curves based on targets.

        Args:
            y_true: Clear ground truth integer evaluation arrays.
            y_prob: Estimated probabilities distributions vector metrics.
            class_names: Identified text categories tracking string targets.
            filename: Targeted file output string.
        """
        logger.info("Generating Receiver Operating Characteristic (ROC) curves...")
        try:
            plt.figure(figsize=(9, 7))

            if len(class_names) == 2:
                # Optimized for Binary Target Analysis splits
                fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
                auc_score = roc_auc_score(y_true, y_prob[:, 1])
                plt.plot(
                    fpr, tpr,
                    label=f"Binary Target Class ROC Curve (AUC = {auc_score:.3f})",
                    color="darkorange",
                    lw=2,
                )
            else:
                # Optimized multi-class mapping logic paths (One-vs-Rest)
                for idx, label in enumerate(class_names):
                    fpr, tpr, _ = roc_curve(y_true == idx, y_prob[:, idx])
                    auc_score = roc_auc_score(y_true == idx, y_prob[:, idx])
                    plt.plot(
                        fpr, tpr, lw=2, label=f"Class: {label} (AUC = {auc_score:.3f})"
                    )

            plt.plot([0, 1], [0, 1], color="navy", linestyle="--", alpha=0.7)
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel("False Positive Rate (FPR)", fontsize=11)
            plt.ylabel("True Positive Rate (TPR)", fontsize=11)
            plt.title(
                "Multi-Class Receiver Operating Characteristic (ROC) Curve Space",
                fontsize=13,
                fontweight="bold",
                pad=15,
            )
            plt.legend(loc="lower right", fontsize=10)
            plt.grid(True, linestyle=":", alpha=0.6)
            plt.tight_layout()
            plt.savefig(filename, dpi=300)
            plt.close()
            logger.info(f"ROC Curves graph saved successfully to {filename}")
        except Exception as e:
            logger.error(f"Failed to generate ROC curve visualizations: {str(e)}")


# ==============================================================================
# MAIN EXECUTION ROUTINE HUB
# ==============================================================================
def main(data_path: str) -> None:
    """Orchestrates ingestion, pipeline transformation, training, and visualization.

    Args:
        data_path: Location of the cleaning outputs file targeted for models.
    """
    logger.info("Initializing main analytical orchestration hub workflow...")

    # Initialize data load execution sequences
    loader = DataLoader(file_path=data_path)
    df = loader.load_data()

    # Step 1: Preprocessing & Data Splits Setup
    pipeline_manager = PropertyPipeline()
    pipeline_manager.build_preprocessor(df)

    # Process regression data structures
    X_train_r, X_test_r, y_train_r, y_test_r = (
        pipeline_manager.prepare_regression_split(df)
    )
    # Re-extract and preserve features maps
    pipeline_manager.extract_feature_names(X_train_r)

    # Process classification data structures
    X_train_c, X_test_c, y_train_c, y_test_c, class_labels = (
        pipeline_manager.prepare_classification_split(df)
    )

    # Validate generated dimensions matching 80/20 goals
    logger.info(f"Regression instances check - Train shape: {X_train_r.shape}")
    logger.info(f"Classification instances check - Train shape: {X_train_c.shape}")

    # Initialize execution engine tracking systems
    trainer = ModelTrainer(
        preprocessor=pipeline_manager.preprocessor, random_state=42
    )

    # Step 2: Train & Tune Regressor
    best_reg_pipeline, reg_metrics = trainer.train_regressor(
        X_train_r, X_test_r, y_train_r, y_test_r
    )

    # Step 3: Train & Tune Classifier
    best_clf_pipeline, y_pred_c, y_prob_c, clf_metrics = trainer.train_classifier(
        X_train_c, X_test_c, y_train_c, y_test_c
    )

    # ==========================================================================
    # FINAL METRIC COMPILATION AND DUMPING
    # ==========================================================================
    print("\n" + "=" * 60)
    print("           ACADEMIC MODEL PERFORMANCE EVALUATION REPORT        ")
    print("=" * 60)

    print("\n[REGRESSION PERFORMANCE METRICS - PROPERTY PRICE]")
    df_reg_metrics = pd.DataFrame(
        {
            "Baseline": [
                reg_metrics["Baseline"]["RMSE"],
                reg_metrics["Baseline"]["MAE"],
                reg_metrics["Baseline"]["R2"],
            ],
            "Tuned": [
                reg_metrics["Tuned"]["RMSE"],
                reg_metrics["Tuned"]["MAE"],
                reg_metrics["Tuned"]["R2"],
            ],
        },
        index=["RMSE", "MAE", "R2"],
    )
    print(df_reg_metrics.round(4))
    print(f"\nBest Regressor Hyperparameters: {reg_metrics['Best_Params']}")

    print("\n" + "-" * 60)
    print("[CLASSIFICATION PERFORMANCE METRICS - PROPERTY USAGE]")
    df_clf_metrics = pd.DataFrame(
        {
            "Baseline": [
                clf_metrics["Baseline"]["Accuracy"],
                clf_metrics["Baseline"]["F1_Weighted"],
            ],
            "Tuned": [
                clf_metrics["Tuned"]["Accuracy"],
                clf_metrics["Tuned"]["F1_Weighted"],
            ],
        },
        index=["Accuracy", "F1 (Weighted)"],
    )
    print(df_clf_metrics.round(4))
    print(f"\nBest Classifier Hyperparameters: {clf_metrics['Best_Params']}")

    print("\nDetailed Academic Classification Report (Tuned Ensemble):")
    print(
        classification_report(
            y_test_c, y_pred_c, target_names=class_labels, digits=4
        )
    )

    # Step 4: Run Isolated Visualizations
    logger.info("Executing plot saving workflows...")
    Visualizer.plot_feature_importance(
        model_pipeline=best_reg_pipeline,
        feature_names=pipeline_manager.transformed_feature_names,
        filename="rf_feature_importances.png",
    )

    Visualizer.plot_confusion_matrix(
        y_true=y_test_c,
        y_pred=y_pred_c,
        class_names=class_labels,
        filename="rf_confusion_matrix.png",
    )

    Visualizer.plot_roc_curves(
        y_true=y_test_c,
        y_prob=y_prob_c,
        class_names=class_labels,
        filename="rf_roc_curves.png",
    )

    logger.info("All workflow pipelines concluded without errors.")

In [2]:
from typing import Optional

def main_with_sampling(data_path: str, rows_to_load: Optional[int] = None) -> None:
    """Orchestrates ingestion, pipeline transformation, training, and visualization with optional sampling during loading."""
    logger.info("Initializing main analytical orchestration hub workflow with sampling...")

    loader = DataLoader(file_path=data_path, nrows=rows_to_load)
    df = loader.load_data()

    import sys
    # Check memory usage of the DataFrame after loading
    logger.info(f"Memory usage of DataFrame: {sys.getsizeof(df) / (1024**2):.2f} MB")

    # No need for in-memory sampling here, as DataLoader already loaded a subset

    # Step 1: Preprocessing & Data Splits Setup
    pipeline_manager = PropertyPipeline()
    pipeline_manager.build_preprocessor(df)

    # Process regression data structures
    X_train_r, X_test_r, y_train_r, y_test_r = (
        pipeline_manager.prepare_regression_split(df)
    )
    pipeline_manager.extract_feature_names(X_train_r)

    # Process classification data structures
    X_train_c, X_test_c, y_train_c, y_test_c, class_labels = (
        pipeline_manager.prepare_classification_split(df)
    )

    logger.info(f"Regression instances check - Train shape: {X_train_r.shape}")
    logger.info(f"Classification instances check - Train shape: {X_train_c.shape}")

    trainer = ModelTrainer(
        preprocessor=pipeline_manager.preprocessor, random_state=42
    )

    # Step 2: Train & Tune Regressor
    best_reg_pipeline, reg_metrics = trainer.train_regressor(
        X_train_r, X_test_r, y_train_r, y_test_r
    )

    # Step 3: Train & Tune Classifier
    best_clf_pipeline, y_pred_c, y_prob_c, clf_metrics = trainer.train_classifier(
        X_train_c, X_test_c, y_train_c, y_test_c
    )

    print("\n" + "=" * 60)
    print("           ACADEMIC MODEL PERFORMANCE EVALUATION REPORT        ")
    print("=" * 60)

    print("\n[REGRESSION PERFORMANCE METRICS - PROPERTY PRICE]")
    df_reg_metrics = pd.DataFrame(
        {
            "Baseline": [
                reg_metrics["Baseline"]["RMSE"],
                reg_metrics["Baseline"]["MAE"],
                reg_metrics["Baseline"]["R2"],
            ],
            "Tuned": [
                reg_metrics["Tuned"]["RMSE"],
                reg_metrics["Tuned"]["MAE"],
                reg_metrics["Tuned"]["R2"],
            ],
        },
        index=["RMSE", "MAE", "R2"],
    )
    print(df_reg_metrics.round(4))
    print(f"\nBest Regressor Hyperparameters: {reg_metrics['Best_Params']}")

    print("\n" + "-" * 60)
    print("[CLASSIFICATION PERFORMANCE METRICS - PROPERTY USAGE]")
    df_clf_metrics = pd.DataFrame(
        {
            "Baseline": [
                clf_metrics["Baseline"]["Accuracy"],
                clf_metrics["Baseline"]["F1_Weighted"],
            ],
            "Tuned": [
                clf_metrics["Tuned"]["Accuracy"],
                clf_metrics["Tuned"]["F1_Weighted"],
            ],
        },
        index=["Accuracy", "F1 (Weighted)"],
    )
    print(df_clf_metrics.round(4))
    print(f"\nBest Classifier Hyperparameters: {clf_metrics['Best_Params']}")

    print("\nDetailed Academic Classification Report (Tuned Ensemble):")
    print(
        classification_report(
            y_test_c, y_pred_c, target_names=class_labels, digits=4
        )
    )

    logger.info("Executing plot saving workflows...")
    Visualizer.plot_feature_importance(
        model_pipeline=best_reg_pipeline,
        feature_names=pipeline_manager.transformed_feature_names,
        filename="rf_feature_importances.png",
    )

    Visualizer.plot_confusion_matrix(
        y_true=y_test_c,
        y_pred=y_pred_c,
        class_names=class_labels,
        filename="rf_confusion_matrix.png",
    )

    Visualizer.plot_roc_curves(
        y_true=y_test_c,
        y_prob=y_prob_c,
        class_names=class_labels,
        filename="rf_roc_curves.png",
    )

    logger.info("All workflow pipelines concluded without errors.")

In [5]:
if __name__ == '__main__':
    TARGET_DATA_PATH = "/content/drive/MyDrive/CSCI323 - project dld transaction Data/cleaned_data.csv"

    # Load full dataset then sample randomly for better representation
    loader = DataLoader(file_path=TARGET_DATA_PATH)
    df_full = loader.load_data()
    df_sampled = df_full.sample(n=50000, random_state=42)

    # Save sampled data temporarily and run pipeline on it
    df_sampled.to_csv("/content/sampled_data.csv", index=False)
    main_with_sampling(data_path="/content/sampled_data.csv", rows_to_load=None)


           ACADEMIC MODEL PERFORMANCE EVALUATION REPORT        

[REGRESSION PERFORMANCE METRICS - PROPERTY PRICE]
          Baseline         Tuned
RMSE  4.207139e+06  8.581849e+06
MAE   2.015423e+05  9.982143e+05
R2    8.945000e-01  5.610000e-01

Best Regressor Hyperparameters: {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}

------------------------------------------------------------
[CLASSIFICATION PERFORMANCE METRICS - PROPERTY USAGE]
               Baseline   Tuned
Accuracy         0.9813  0.9814
F1 (Weighted)    0.9814  0.9815

Best Classifier Hyperparameters: {'model__n_estimators': 50, 'model__min_samples_split': 2, 'model__max_features': 'sqrt', 'model__max_depth': None}

Detailed Academic Classification Report (Tuned Ensemble):
              precision    recall  f1-score   support

  Commercial     0.9563    0.9230    0.9393       948
       Other     0.8538    0.8953    0.8741       535
 Residential     0.